# Adaptive N-AFC staircase — configured by one file

An **adaptive staircase** repeatedly presents an N-AFC trial, making the task
harder after correct answers and easier after wrong ones, so it homes in on the
participant's **threshold**.

To keep this simple, the **entire experiment is configured in a single file:**
[`configs/staircase_n_afc.yml`](../configs/staircase_n_afc.yml). The only thing
configured elsewhere is the shared look of the UIs
([`configs/design.yml`](../configs/design.yml)).

That one file is read by three consumers, each taking the keys it needs:

| block in the YAML | used by | configures |
|---|---|---|
| `SoundDevice:` | the audio handler | which WAV plays for each stimulus id |
| `test:` / `ui:` | the N-AFC window | #alternatives, shuffling, window size, wording |
| `trial:` | this notebook | what one trial is (the odd-one-out design) |
| `staircase:` | this notebook | the adaptive rule |

**To design your own experiment, edit that YAML file** — you should not need to
change any Python below. If you haven't installed the project yet, run this
from the repo root: `pip install -e .`

In [8]:
import whispy
from pathlib import Path
from whispy.interfaces import SoundDevice
from whispy.ui import NAFC
from whispy.utils import read_config

config_path = Path('..') / 'configs' / 'staircase_n_afc.yml'  # <- the one file you edit
stimuli_dir = Path('stimuli_staircase')                                 # <- folder with your WAVs

cfg = read_config(config_path)
handler = SoundDevice(config_path, stimuli_dir, loop=False)   # reads the SoundDevice: block
print('available stimulus ids:', list(handler.stimuli.keys()))

available stimulus ids: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


## The configuration

Open [`configs/staircase_n_afc.yml`](../configs/staircase_n_afc.yml) to see
every setting with inline comments. The cell below prints the parts that define
the experiment so you can see your current values at a glance.

Key things you'll typically change:

- **`trial.task`** — the prompt shown on top of the window.
- **`trial.standard` / `trial.levels`** — the reference stimulus and the target
  ids ordered *hardest → easiest*.
- **`test.n_choices`** — how many intervals per trial (the *N* in N-AFC).
- **`staircase.n_up` / `n_down`** — the adaptive rule (1-up/2-down ≈ 70.7%).
- **`staircase.max_reversals` / `max_trials`** — when the run stops.

In [9]:
print('test     :', cfg['test'])
print('ui       :', cfg['ui'])
print('trial    :', cfg['trial'])
print('staircase:', cfg['staircase'])

test     : {'n_choices': 3, 'shuffle_choices': True}
ui       : {'window_size': 'fullscreen', 'submit_hint': 'Press the number keys (or click) to hear each interval, then press Enter to submit your choice.', 'submit_button_text': 'Submit choice'}
trial    : {'task': 'Which interval is the **odd one out**?', 'standard': 1, 'levels': [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], 'block_name': 'Staircase', 'section_name': '3-AFC'}
staircase: {'n_up': 1, 'n_down': 2, 'step': 1, 'start_index': -1, 'max_reversals': 6, 'max_trials': 4, 'reversals_for_threshold': 4}


## Wire the config to a staircase

These few lines just translate the config into a `build_screen(level)` callback
(which turns a difficulty level into an N-AFC trial) and a `whispy.Staircase`.
You normally don't edit this — it is driven entirely by the values above.

The trial is an **odd-one-out**: `n_choices - 1` intervals play the `standard`
and one plays the target `level` (the correct answer); `NAFC` shuffles the
buttons. The participant must listen to every interval before submitting.

In [10]:
trial = cfg['trial']
levels = trial['levels']
n_intervals = int(cfg['test']['n_choices'])

def build_screen(level):
    return {
        'test': [trial['standard']] * (n_intervals - 1) + [level],
        'correct': level,
        'task': trial['task'],
        'block': 0,
        'section': 0,
        'block_name': trial.get('block_name', 'Staircase'),
        'section_name': trial.get('section_name', 'N-AFC'),
    }

sc_kwargs = dict(cfg['staircase'])
if sc_kwargs.get('start_index', 0) < 0:          # allow -1 = easiest (last) level
    sc_kwargs['start_index'] += len(levels)

staircase = whispy.Staircase(levels, build_screen=build_screen, **sc_kwargs)
print('start level:', staircase.current_level, '| finished:', staircase.finished)

start level: 12 | finished: False


## Validate your configuration without a GUI (dry run)

Run this **before** the live experiment below to sanity-check your
`staircase:` settings: it replaces the window with a simulated observer that
answers correctly with a per-level probability, then reports whether the run
converges and collects enough reversals/trials. Tune the YAML and re-run this
cell — no clicking required. (It reuses `cfg`, `levels` and `build_screen`
from the cells above, so run those first.)

In [11]:
import numpy as np

rng = np.random.default_rng(0)

# A simulated observer: probability of a correct response at each target
# level. Derived from your `levels` so it always covers every level no matter
# how many there are. `levels` is ordered hardest -> easiest, so accuracy
# ramps from just above chance (1/n_choices) up to near-certain. Replace this
# dict with your own numbers to model a specific listener.
chance = 1.0 / int(cfg['test']['n_choices'])
p_values = np.linspace(chance + 0.05, 0.98, len(levels))
p_correct = {level: float(p) for level, p in zip(levels, p_values)}

def simulated_trial(screen):
    return bool(rng.random() < p_correct[screen['correct']])

sim_kwargs = dict(cfg['staircase'])
if sim_kwargs.get('start_index', 0) < 0:
    sim_kwargs['start_index'] += len(levels)
sim = whispy.Staircase(levels, build_screen=build_screen, **sim_kwargs)

sim_results = sim.run(simulated_trial)
print('simulated p(correct):', {k: round(v, 2) for k, v in p_correct.items()})
print('simulated trials   :', len(sim_results))
print('simulated reversals:', sim.n_reversals)
print('simulated threshold:', sim.threshold())
sim_results

simulated p(correct): {2: 0.38, 3: 0.44, 4: 0.5, 5: 0.56, 6: 0.62, 7: 0.68, 8: 0.74, 9: 0.8, 10: 0.86, 11: 0.92, 12: 0.98}
simulated trials   : 4
simulated reversals: 0
simulated threshold: None


,trial,level_index,level,correct,step,reversal
0,1,10,12,True,,False
1,2,10,12,True,down,False
2,3,9,11,True,,False
3,4,9,11,True,down,False


## Run it

`staircase.run(run_trial)` drives the whole loop until it stops. `run_trial`
reuses **one window** for the entire experiment (the first trial opens it, later
trials swap their content into the same window via `parent=host`), so there is
no reload between trials and it stays fullscreen. The window is closed in a
`finally` block when the staircase finishes.

In each trial: click a button to hear that interval, listen to **all** of them,
select the odd one out, then press **Submit**.

In [12]:
host = None

def run_trial(screen):
    global host
    naf = NAFC(screen=screen, stimuli_handler=handler,
               n_afc_config=config_path, blocking=True, parent=host)
    if host is None:
        host = naf            # first trial owns the shared window
    return bool(naf.get_results()['correct_bool'].iloc[0])

try:
    results = staircase.run(run_trial)
finally:
    if host is not None:
        host.close()

results

,trial,level_index,level,correct,step,reversal
0,1,10,12,True,,False
1,2,10,12,True,down,False
2,3,9,11,True,,False
3,4,9,11,True,down,False


## Results & threshold

`staircase.get_results()` (returned above) has one row per trial: the level
presented, whether it was correct, the step taken, and whether it was a
reversal. `threshold()` averages the final reversal levels.

In [13]:
print('trials         :', len(results))
print('reversals      :', staircase.n_reversals)
print('reversal levels:', staircase.reversal_levels())
print('threshold      :', staircase.threshold())
results

trials         : 4
reversals      : 0
reversal levels: []
threshold      : None


,trial,level_index,level,correct,step,reversal
0,1,10,12,True,,False
1,2,10,12,True,down,False
2,3,9,11,True,,False
3,4,9,11,True,down,False


## Save the results

Write the per-trial `results` table to a timestamped CSV in a `results/`
folder (created next to this notebook) so each run is kept for later analysis.

In [16]:
from datetime import datetime

results_dir = Path('results')          # CSVs are written next to this notebook
results_dir.mkdir(exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results_path = results_dir / f'staircase_n_afc_{timestamp}.csv'
results.to_csv(results_path, index=False)
print('saved results to', results_path.resolve())

saved results to /Users/tomstrobl/Documents/GitHub/whispy/examples/results/staircase_n_afc_20260622_114133.csv
